# A paper snowflake, as polygons

`preview.webp` is a photo of a printed snowflake: dark ink on paper, yellow around it. This
notebook turns it into an SVG made only of straight segments:

1. split ink, paper and yellow, and look at the connected components;
2. test the polygon fit on a synthetic **dart** (a concave quadrilateral) where the answer is
   known;
3. find the **symmetry** of the flake (dihedral, order 12) and average the image over it;
4. fit one polygon per orbit of components, make it exactly symmetric, copy it around;
5. write `out/snowflake.svg` and compare it with the scan.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import SVG, display
from skimage.measure import approximate_polygon

import flocons
from flocons import geometry as G

print("browser" if flocons.IN_BROWSER else "local")

## 1. Ink, paper, yellow and the connected components

Three luminance classes (multi-Otsu): ink, yellow, paper. The ink is one connected component
(8-connectivity); its holes are the components of the rest that do not reach the border.

In [ ]:
rgb = G.load_preview()
seg = G.segment(rgb)
black = seg["black"]
outside, holes, n_holes = G.outside_and_holes(black)
hole_area = np.bincount(holes.ravel())[1:]
print("image", rgb.shape, "thresholds", np.round(seg["thresholds"], 3))
print("ink components:", seg["n_components"], "| ink pixels:", int(black.sum()))
print("ink colour", G.hex_color(seg["ink_rgb"]), "paper", G.hex_color(seg["paper_rgb"]))
print(
    "holes:",
    n_holes,
    "| area min / median / max:",
    hole_area.min(),
    int(np.median(hole_area)),
    hole_area.max(),
)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5.6))
ax[0].imshow(rgb)
ax[0].set_title("scan")
shown = np.where(black, 0, holes % 9 + 1)  # one colour per hole (9 cycling), ink black
shown = np.ma.masked_where(outside, shown)
ax[1].imshow(shown, cmap="tab10", vmin=0, vmax=10, interpolation="nearest")
ax[1].set_title(f"ink (1 component), {n_holes} holes")
for a in ax:
    a.set_axis_off()
plt.tight_layout()
plt.show()

## 2. The fit, tested on a dart

A dart: four vertices, one of them reentrant. It is drawn like a scan (pixel coverage, a little
blur and noise) at two sizes, the larger about twice a hole of the flake, the smaller about
the size of the smallest holes. Then:

- **Douglas-Peucker** on the 0.5 iso-contour finds the corners, but puts them *on* the raster
  outline, which rounds every corner: the vertices land inside the true shape;
- **refit**: each contour point goes to its nearest edge, each edge becomes the straight line
  fitted to its points (leaving out the ~2 px near the corners, where the rounding is), and each
  vertex moves to the intersection of its two lines. Corners come back sharp. Vertices where the
  outline barely turns are dropped, tiny edges merged; repeated a few times.

In [ ]:
dart = np.array([[50, 12], [78, 80], [50, 58], [22, 80]], float)
tests = []
for scale, shape in [(1.0, (95, 100)), (0.4, (40, 42))]:
    truth = dart * scale + 0.37  # off the pixel grid
    img = G.render_polygon(truth, shape)
    cnt = max(G.contours_xy(img), key=len)
    dp = approximate_polygon(np.vstack([cnt, cnt[:1]]), 1.5)[:-1]
    fit = G.fit_polygon(cnt)
    tests.append((scale, truth, img, dp, fit))
    print(
        f"scale {scale}: vertices DP {len(dp)}, refit {len(fit)}; "
        f"vertex error (px) DP {G.vertex_error(dp, truth).round(2)}, "
        f"refit {G.vertex_error(fit, truth).round(2)}; rms {G.fit_error(fit, cnt):.3f} px"
    )

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
for a, (scale, truth, img, dp, fit) in zip(ax, tests, strict=True):
    a.imshow(img, cmap="gray_r")
    for poly, style, label in [
        (truth, "y-", "truth"),
        (dp, "r--", "Douglas-Peucker"),
        (fit, "c-", "refit"),
    ]:
        closed = np.vstack([poly, poly[:1]])
        a.plot(closed[:, 0], closed[:, 1], style, lw=1.5, marker="o", ms=4, label=label)
    a.set_title(f"dart, scale {scale}")
    a.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## 3. The symmetry

The flake looks hexagonal with mirrors. The centre is where the image overlaps best with its
copies rotated by 60 and 120 degrees; the mirror angle is the one where it overlaps best with
its reflection (searched on a 3x smaller image, then refined). The table gives the overlap
(soft intersection over union) of the scan with each of the 12 elements of the group D6.

In [ ]:
sym = G.find_symmetry(seg["soft"])
c, theta, group = sym["center"], sym["theta"], sym["group"]
names = [f"rot {60 * k:3d}" for k in range(6)] + [
    f"mirror {np.degrees(theta) + 30 * k:5.1f}" for k in range(6)
]
print("centre", c.round(2), "| arm axis", round(np.degrees(theta), 2), "deg (from +x, clockwise)")
for name, score in zip(names, sym["scores"], strict=True):
    print(f"  {name:14s} overlap {score:.3f}")

The overlaps are 0.92-0.96 for every element: the flake is D6-symmetric up to the printing and
the photo (strokes are ~12 px wide, so a 0.5 px shift already costs a few percent). Averaging
the image over the 12 elements keeps what all copies agree on.

In [ ]:
sym_img = G.symmetrize_image(seg["soft"], group, c)
r = 400
fig, ax = plt.subplots(1, 2, figsize=(10, 5.6))
ax[0].imshow(seg["soft"], cmap="gray_r")
for k in range(6):
    a = theta + k * np.pi / 6
    style = "r-" if k % 2 == 0 else "b--"
    ax[0].plot(
        [c[0] - r * np.cos(a), c[0] + r * np.cos(a)],
        [c[1] - r * np.sin(a), c[1] + r * np.sin(a)],
        style,
        lw=0.8,
    )
ax[0].set_title("mirrors: arms (red), between (blue)")
ax[1].imshow(sym_img, cmap="gray_r")
ax[1].set_title("average over D6")
for a in ax:
    a.set_xlim(0, rgb.shape[1])
    a.set_ylim(rgb.shape[0], 0)
    a.set_axis_off()
plt.tight_layout()
plt.show()

## 4. One polygon per orbit

The averaged image has one outline and holes in orbits of 6 (on a mirror) or 12 (off the
mirrors); each orbit has one member in the fundamental wedge (between an arm axis and the next
mirror, 30 degrees on), its **representative**. The averaged image is only used to name the
orbits: its corners are rounder than the scan's (12 slightly misaligned copies), and fitting it
gave some holes 7 or 8 vertices instead of 4 or 6.

So every component of the **scan** is fitted on its own, as in section 2. Then, for each
representative, the group elements that carry a scanned hole onto it fold that hole's vertices
there: 12 copies per orbit (an orbit of 6 holes on a mirror is folded by 2 elements each).
The copies are shifted onto the representative (their centroids are up to ~1.4 px apart),
clustered (4 px across at most), and a vertex is kept where at least half the copies have one.
Last, the polygon is made exactly symmetric: only the vertices in one fundamental region of
its stabilizer are kept (one side of its mirror; for the outline, the 30-degree wedge), those
within 1.5 px of a mirror are put on it, and the rest is their mirror images.

In [ ]:
holes_kept = int((hole_area > 10).sum())
print(
    f"holes in the scan: {n_holes}, of which {n_holes - holes_kept} specks of <= 10 px "
    f"(areas {sorted(hole_area[hole_area <= 10].tolist())}) are left out"
)
flake = G.fit_flake(seg["soft"], sym_img, group, c, theta)
rings = flake["rings"]
print(
    f"components fitted in the scan: {len(flake['raw_contours'])} | "
    f"rings in the model: {len(rings)} (outline + {len(rings) - 1} holes)"
)
print("orbit       r(px) phi(deg) vertices scan-fits stabilizer copies folded dropped  rms(px)")
for i, row in enumerate(flake["rows"]):
    name = "outline" if i == 0 else f"hole {i:2d}"
    where = "      -      -" if i == 0 else f"{row['r']:7.1f} {round(row['phi'], 1) + 0.0:7.1f}"
    fits = "/".join(map(str, row["raw_vertices"]))
    print(
        f"{name:8s} {where} {row['vertices']:8d} {fits:>9s} {row['stabilizer']:10d}"
        f" {row['copies']:6d} {row['folded']:6d} {row['dropped']:7d} {row['rms']:8.3f}"
    )

*scan-fits*: vertex counts of the scanned copies fitted one by one; *dropped*: clusters held by
fewer than half the copies (a corner cut in two on a few copies); *rms*: distance from the
folded scan contours to the model polygon.

In [ ]:
colors = plt.cm.tab20(np.arange(len(flake["reps"])) % 20)
fig, ax = plt.subplots(1, 2, figsize=(10, 5.6))
ax[0].imshow(rgb)
for rep, col in zip(flake["reps"], colors, strict=True):
    closed = np.vstack([rep, rep[:1]])
    ax[0].fill(closed[:, 0], closed[:, 1], color=col, alpha=0.8)
outer = np.vstack([flake["outer"], flake["outer"][:1]])
ax[0].plot(outer[:, 0], outer[:, 1], "c-", lw=0.8)
for k in (0, 1):
    a = theta + k * np.pi / 6
    ax[0].plot([c[0], c[0] + r * np.cos(a)], [c[1], c[1] + r * np.sin(a)], "k:", lw=0.8)
ax[0].set_title("fundamental wedge: one hole per orbit")
ax[0].set_axis_off()
ax[1].imshow(rgb)
for ring in rings:
    closed = np.vstack([ring, ring[:1]])
    ax[1].plot(closed[:, 0], closed[:, 1], "c-", lw=1)
    ax[1].plot(ring[:, 0], ring[:, 1], "r.", ms=3)
ax[1].set_xlim(c[0] - 90, c[0] + 90)
ax[1].set_ylim(c[1] - 230, c[1] - 440)  # image rows go down
ax[1].set_title("top arm: model on the scan")
plt.tight_layout()
plt.show()

## 5. The SVG

One `<path>` with the even-odd fill rule: the outline, then the holes. Only straight segments,
no stroke: the corners are exactly the fitted ones.

In [ ]:
Path("out").mkdir(exist_ok=True)
svg = G.to_svg(rings, fill=G.hex_color(seg["ink_rgb"]))
Path("out/snowflake.svg").write_text(svg)
n_vertices = sum(len(p) for p in rings)
print(f"out/snowflake.svg: {len(svg)} bytes, {len(rings)} rings, {n_vertices} vertices")
display(SVG(svg))

**How close.** The model rasterized at the scan's resolution, against the scan's ink mask and
against the D6 average. Grey: both; red: ink the model misses; blue: model where the scan has
no ink. What is left is mostly the scan's own departure from symmetry.

In [ ]:
model = G.rasterize(rings, black.shape)
sym_mask = sym_img > 0.5
print(
    f"IoU model vs scan: {G.iou(model, black):.4f}"
    f" | model vs D6 average: {G.iou(model, sym_mask):.4f}"
    f" | scan vs D6 average: {G.iou(black, sym_mask):.4f}"
)
diff = np.ones(black.shape + (3,))
diff[black & model] = 0.75
diff[black & ~model] = [0.85, 0.1, 0.1]
diff[model & ~black] = [0.1, 0.35, 0.9]
fig, ax = plt.subplots(figsize=(6, 6.9))
ax.imshow(diff)
ax.set_title("both (grey), scan only (red), model only (blue)")
ax.set_axis_off()
plt.tight_layout()
plt.show()